In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ["OPEN_API_KEY"]=os.getenv("OPEN_API_KEY")
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

### Summarization Middleware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)



In [7]:
### Run with thread id

config={"configurable":{"thread_id":"test-1"}}

In [8]:
#  Alternative test data
questions=[
    "what is 2+2",
    "what is 10*5",
    "what is the highest rated imdb movie",
    "which is artificial interllignece",
    "what is 4*54",
    "what is a leap year",
    "which is best university to do masters in coumpter science"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='f3c8b19d-a3c8-4147-92db-668f099d9a5d'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks "what is 2+2". Simple. Provide answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 77, 'total_tokens': 115, 'completion_time': 0.03918602, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.003693451, 'prompt_tokens_details': None, 'queue_time': 0.310093757, 'total_time': 0.042879471}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0946f-6deb-7721-8c70-77ac032079d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 38, 'total_tokens': 115, 'output_token_details': {'reasoning': 19}})]}
messages:2
Mes

### Token size

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa , pool, gym
    2. City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 start,$75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)


config={"configurable":{"thread_id":"test-1"}}


def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//6

In [15]:
cities=["Paris","London","Tokyo", "New York", "Dubai","Singapore"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"Find hotels in {city}")]
    },
    config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens}, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~199, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='7296df9a-e0d0-4f9d-970c-3e5fceddc68d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function search_hotels that takes {city: string}. We should call that.', 'tool_calls': [{'id': 'fc_e78305e4-7098-48b7-98eb-938a5e8453e8', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 129, 'total_tokens': 182, 'completion_time': 0.055048622, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.00732609, 'prompt_tokens_details': None, 'queue_time': 0.210519228, 'total_time': 0.062374712}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m1ye8z0zefbrpykxk631rver` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6681, Requested 1516. Please try again in 1.4775s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Human in the Loop MiddleWare